# 01 対話と記録

**目的**：自分の質問で対話し、質問や入力範囲を変えた理由と、自分の理解の変化を記録します。通常の演習は個人5分・相談と対話5分・記録5分です。GPUは不要です。

共通課題ではPaper2Agent論文の版・指定範囲・3問が固定されています。別の論文を使うHomeworkは、共通課題が終わってから最後の節で切り替えます。

**通信と費用**：有料APIを呼ぶのは後半の送信セルだけです。既定値は送信OFFです。講義用の接続設定・利用許可・個別キーは教員の案内に従います。公開されている空設定では送信できません。

期間全体で1人10 USD分を確保し、基本的な使い方を想定しています。学生全体にも使用量の上限があります。利用期限は11月14日23:59（日本時間）。キーや接続情報をNotebook本文、出力、Slack、GitHub、他のAIへ載せないでください。

**保存**：自分の説明とAIの生回答を分け、最後にJSONLとMarkdownをダウンロードします。保存済みファイルは上書きしません。

## Colabで最初に行うこと

次のセルの `PREPARE_COLAB` を **True** にして実行します。公開リポジトリの固定コミットを取得し、SHA-256が一致したときだけ展開します。取得と依存のインストールには通信が必要ですが、有料APIは呼びません。既存の教材ファイルが変更されている場合は、黙って使い続けずエラーになります。

実行コードはセルを開いて確認できます。接続設定とAPIキーは教員の案内に従い、Notebook本文には保存しないでください。Google Driveへの自動保存・共有は行いません。


In [ ]:
# Colabで初回だけTrueにする。固定版の取得と依存のインストール。APIは呼ばない。
PREPARE_COLAB = False  #@param {type:"boolean"}
RELEASE_COMMIT = "a9079daf145ac8ddf48376257db8e21add906a90"
RELEASE_SHA256 = "291cadf06914ca5bc9b077a95bda0bea0e4ebee0fe6c03203fbf0166a249c07f"

# BEGIN VERIFIED BOOTSTRAP
"""Notebook冒頭へ埋め込む標準ライブラリだけの固定版取得処理。"""
from pathlib import Path, PurePosixPath
from io import BytesIO
from hashlib import sha256
from urllib.request import urlopen
import json
import re
import stat
import zipfile


def fetch_course(destination: Path, commit: str, expected_sha256: str) -> Path:
    """固定コミット・ハッシュ一致・安全な相対パスを確認して新しい教材フォルダを作る。"""
    if not re.fullmatch(r'[0-9a-f]{40}', commit) or not re.fullmatch(r'[0-9a-f]{64}', expected_sha256):
        raise ValueError('検証済みのコミットとSHA-256が設定されていません。')
    destination = destination.resolve()
    if destination.exists():
        marker = destination / '.course-release.json'
        saved = json.loads(marker.read_text()) if marker.is_file() else {}
        if saved.get('commit') != commit or saved.get('sha256') != expected_sha256 or not saved.get('files'):
            raise ValueError('既存フォルダの版が一致しません。別の空フォルダを指定してください。')
        for name, digest in saved['files'].items():
            target = (destination / name).resolve()
            if not target.is_relative_to(destination) or not target.is_file() or sha256(target.read_bytes()).hexdigest() != digest:
                raise ValueError('展開済み教材に欠落・変更があります。新しいフォルダへ取得してください。')
        return destination
    url = f'https://codeload.github.com/humansys-lab/multimodal-ai-paper-reading/zip/{commit}'
    with urlopen(url, timeout=40) as response:
        archive = response.read(20_000_001)
    if len(archive) > 20_000_000 or sha256(archive).hexdigest() != expected_sha256:
        raise ValueError('教材ZIPのサイズまたはハッシュが一致しません。展開を中止しました。')
    unpack_course(archive, destination, commit)
    files = {p.relative_to(destination).as_posix(): sha256(p.read_bytes()).hexdigest() for p in destination.rglob('*') if p.is_file()}
    (destination / '.course-release.json').write_text(json.dumps({'commit': commit, 'sha256': expected_sha256, 'files': files}))
    return destination


def unpack_course(archive: bytes, destination: Path, commit: str) -> None:
    """全メンバーの確認後に展開する。既存ファイル・リンク・不正パスを拒否する。"""
    destination = destination.resolve()
    if destination.exists():
        raise ValueError('展開先はまだ存在しないフォルダを指定してください。')
    prefix = f'multimodal-ai-paper-reading-{commit}'
    with zipfile.ZipFile(BytesIO(archive)) as bundle:
        members = []; names = set(); total = 0
        for entry in bundle.infolist():
            parts = PurePosixPath(entry.filename).parts
            if not parts or parts[0] != prefix or '..' in parts or '\\' in entry.filename or stat.S_ISLNK(entry.external_attr >> 16):
                raise ValueError('教材ZIPに不正なパスまたはリンクがあります。')
            if entry.is_dir():
                continue
            if len(parts) < 2:
                raise ValueError('教材ZIPのフォルダ構造が不正です。')
            relative = Path(*parts[1:])
            target = (destination / relative).resolve()
            if not target.is_relative_to(destination) or relative.as_posix() in names:
                raise ValueError('教材ZIPの範囲外・重複パスを検出しました。')
            names.add(relative.as_posix()); total += entry.file_size
            if total > 50_000_000:
                raise ValueError('教材ZIPの展開後サイズが上限を超えています。')
            members.append((entry, target))
        required = {'src/seminar_lab/__init__.py', 'config/course.yaml', 'materials/manifest.yaml', 'requirements-colab.txt'}
        if not required <= names:
            raise ValueError('教材ZIPに必要なコード・設定がありません。')
        destination.mkdir(parents=True)
        for entry, target in members:
            target.parent.mkdir(parents=True, exist_ok=True)
            with target.open('xb') as handle:
                handle.write(bundle.read(entry))
# END VERIFIED BOOTSTRAP

import sys, subprocess
if Path('/content').is_dir():
    ROOT = Path('/content') / ('seminar-' + RELEASE_COMMIT[:12])
    if not PREPARE_COLAB and not ROOT.exists():
        raise RuntimeError('新規ColabではPREPARE_COLABをTrueにして、このセルを実行してください。')
    ROOT = fetch_course(ROOT, RELEASE_COMMIT, RELEASE_SHA256)
    if PREPARE_COLAB:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--require-hashes',
                        '--cache-dir', str(ROOT / 'build/pip-cache'),
                        '-r', str(ROOT / 'requirements-colab.txt')], check=True)
else:
    if PREPARE_COLAB:
        raise RuntimeError('PREPARE_COLABはColab用です。ローカルではFalseにし、仮想環境を使ってください。')
    ROOT = Path.cwd().resolve()
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent
    if not (ROOT / 'src/seminar_lab/__init__.py').is_file():
        raise RuntimeError('教材リポジトリ、またはそのnotebooksフォルダで実行してください。')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from seminar_lab.config import load_yaml, material_context
from seminar_lab.client import Session, LivePermission, OpenAITransport, output_text
from seminar_lab.ui import activity_view, preview_payload, save_pair
from seminar_lab.records import new_record, load_records
from seminar_lab.inputs import text_input, image_input, pdf_input
course = load_yaml(ROOT / 'config/course.yaml')
manifest = load_yaml(ROOT / 'materials/manifest.yaml')
print('ローカル設定読込み完了。まだAPIへ送信していません。')

## 対話の準備（送信なし）
実行経路の設定は教員から受け取る非公開ファイルを指定します。公開の空設定では送れません。
講義用はcourse_proxy、外部自習はdirect_openaiを明示的に選びます。モデル、対応設定、保持方針、講義の予算停止は授業用の接続で検証してから設定します。

初回質問欄は空欄。共通system/developer指示は追加していません。講義後の読み直しは、新しい会話（new）から始めます。

In [ ]:
from uuid import uuid4
runtime_path = ROOT / 'config/runtime.example.yaml'  # 許可された非公開設定へ変更
runtime = load_yaml(runtime_path)
activity_id = 'P1'  # P2a等へ切替可。共通課題の素材は自動的にP01
print(activity_view(course, manifest, activity_id))
if 'session' not in globals():
    session = Session()
if 'run_id' not in globals():
    run_id = str(uuid4())  # このセルを再実行してもIDは維持する
question = ''
conversation_mode = 'new'  # 'continue'は成功済み会話の継続
parameters = {'max_output_tokens': 400}  # 提案値。経路の確認済み範囲以外は拒否
inputs = []

In [ ]:
# P01の許可済み範囲を自分で抜粋する。質問の代わりとなる完成指示は追加しない。
source_excerpt = ''
source_location = ''
inputs = [text_input(source_excerpt, source_location)] if source_excerpt.strip() else []

## 送信前の確認
実際の質問・全履歴を確認します。画像/PDFのbase64だけ表示上省略されるので、元ファイルの見た目も開いて確認してください。入力方式の変更や履歴切捨ては自動で行いません。

In [ ]:
if question.strip():
    context = (homework_context if activity_id == 'HW1' else material_context(course, manifest, activity_id))
    print('資料と版・指定範囲:', context['material_id'], context.get('source_version'), context.get('assigned_scope'))
    print('実入力の由来:', [x.provenance for x in inputs])
    print(preview_payload(session.preview(runtime, question, inputs, conversation_mode, parameters)))
else:
    print('質問欄は空です。自分の問いを入力してください。送信していません。')

## 送信（有料・許可後だけ）
教員等の明示許可を受け、対象・USD上限・認証経路・回数を記録したLivePermissionを別途設定します。キーがあるだけでは許可になりません。講義用本キーは使いません。`SEND=True`だけではpermission未設定のため送れません。

タイムアウト時の課金は不明。失敗時に別モデルや別入力へ切り替えません。再試行は自動0回。回数枠は費用上限の保証ではありません。

In [ ]:
SEND = False
if 'permission' not in globals():
    permission = None  # 教員が案内した対象・上限・認証経路の許可を別セルで設定
if SEND:
    if not isinstance(permission, LivePermission):
        raise ValueError('対象・上限・認証経路を含む許可が必要です。')
    context = (homework_context if activity_id == 'HW1' else material_context(course, manifest, activity_id))
    # キー入力より先に、素材・設定・送信内容を検証
    session.preview(runtime, question, inputs, conversation_mode, parameters)
    from getpass import getpass
    transport = OpenAITransport(runtime, getpass('許可された個別APIキー（表示・保存しません）: '), permission)
    try:
        result = session.run(runtime=runtime, context=context, prompt=question, inputs=inputs,
                             mode=conversation_mode, parameters=parameters, run_id=run_id, transport=transport)
    finally:
        transport.close()
    print('状態:', result['status'])
    if result.get('error_message'):
        print(result['error_message'])
    print(output_text(result))
    print('費用:', result['cost_value'] if result['cost_value'] is not None else '不明')
else:
    print('送信OFF。APIは呼び出していません。')

## 本人の説明と保存
生回答はresponse_rawに残し、本人の説明・修正文は別に書きます。保存は明示操作で、上書きしない名前を作ります。Colabを閉じる・再起動する前にJSONLとMarkdownの両ファイルをダウンロードしてください。出力には対話があるので公開しません。
個人記録には、AI利用前の自分の説明、AIの生回答、原文確認後の自分の説明、問い・入力・履歴の変更、根拠の位置を残してください。

In [ ]:
# Noneは前の記入を保持。文字列を入れた欄だけ更新する。
annotations = {
    'student_explanation': None,
    'evidence_location': None,
    'student_revision': None,
    'unresolved_point': None,
    'next_change': None,
    'change_reason': None,
}
SAVE = False
if SAVE:
    if not session.records:
        raise ValueError('保存する実行記録がありません。先に対話を実行してください。')
    if any(value is not None and not isinstance(value, str) for value in annotations.values()):
        raise ValueError('記入欄は文字列かNoneにしてください。')
    session.records[-1].update({key: value for key, value in annotations.items() if value is not None})
    paths = save_pair(session.records, ROOT / 'outputs')
    print([p.name for p in paths])
    assert load_records(paths[0]) == session.records
    assert load_records(paths[1]) == session.records
    # Colabのファイル欄からoutputs内の2ファイルをダウンロード。
else:
    print('保存は未実行です。前の記入内容も変更していません。')


## 次の質問・意図した反復
同じrun_idで送信セルを再実行すると拒否します。次の質問または確認済みの再試行をする場合だけ、次のセルを有効にして新しいrun_idを作り、質問・新規/継続・プレビューへ戻ります。別の学生は新しいランタイムを使います。

In [ ]:
START_NEXT_RUN = False
if START_NEXT_RUN:
    run_id = str(uuid4())
    print('新しいrunを準備しました。質問・新規/継続・プレビューを見直してください。')

## 共通課題終了後のHomework（別セッションへの明示的な切替）
HW1は全員が別の掲載原著論文を選びます。APIの利用は必須ではありません。P01のR0～R3を保存してから使います。この節でのみ素材を選べます。学生選定資料の確認は教員採用・公開許可へ読み替えません。新しいSessionとhomework段階へ切り替え、上の質問・プレビュー・送信・保存セルを再利用します。
共通の「対話の準備」セルを再実行するとP1へ戻るため、Homework中はそこを実行せず質問と新規/継続を設定してください。

対象誌はNature／Science本誌2誌に確定（Q3 YES）。個別論文の入力条件・認証経路・有料実行許可等を満たすまで送信しません。`local_material`にはid、source_url（出版社の掲載ページ）、doi、bibliography（title/authors/journal）、source_version、assigned_scope、publication_status=published、article_type=research、open_access=trueを、確認できた情報で記入します。`eligibility_review`と`input_review`はそれぞれstatusとreviewerを残してください。未確認をconfirmedで埋めないでください。
P01の任意の復習はP01_REVIEWで別記録にし、HW1の代わりにしません。詳細はmaterials/papers/extensions.md。

In [ ]:
START_HOMEWORK = False
if START_HOMEWORK:
    homework_selection = ''  # 条件を確認したE01等、またはU_<local_id>を選ぶ。P01はHW1の対象外
    local_material = None  # Uの場合は下の項目を手元で記入。空のまま送信しない
    homework_context = material_context(course, manifest, 'HW1', homework_selection, local_material)
    session = Session()  # 共通課題の保存後に新しい状態を作る
    activity_id = 'HW1'
    run_id = str(uuid4())
    conversation_mode = 'new'
    question = ''
    inputs = []
    print(activity_view(course, manifest, 'HW1'))
    print('記録段階:', homework_context['phase'])